# MNIST Neural Network Demo
## Week 4: Introduction to Neural Networks

This notebook demonstrates:
1. What MNIST data looks like
2. Loading a pre-trained neural network
3. Testing the model on real digits
4. Understanding model predictions
5. Exploring when the model fails

**Important**: We're NOT training a model today - just using one!

### If you want to do this yourself, here are some pre-lab set up to follow:

1. Activate your conda in your command line program:

    `conda activate coding3`

2. Install PyTorch and Dependencies (The Stability Fix)
This step uses the highly stable conda install method, which should prevent the kernel crashes.

In command line, let's install PyTorch and main dependencies using the official Conda channel:

    conda install pytorch torchvision matplotlib numpy -c pytorch

3. Launch the Jupyter Notebook
After the installation completes, launch Jupyter. Make sure you navigate to the correct folder first by using `cd` in command line to navigate to the folder containing the notebook:

    `cd path/to/your/notebook/folder`

Then, start Jupyter. Type the following into your command line program:

    `jupyter notebook`

Once the Jupyter notebook opens in your browser, the kernel is already loaded with all the packages. You should now be able to run all cells, including the import cells without any errors or crashes.

## Part 1: Setup and Imports

In [ ]:
# Install required packages (only run this once)
!pip install torch torchvision matplotlib numpy --quiet

print("Packages installed!")

In [ ]:
# Import everything we need
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\nReady to go!")

## Part 2: Understanding MNIST Data

Let's first look at what MNIST digits actually look like:

In [ ]:
# Download MNIST dataset
# This will create a 'data' folder and download the files (only happens once)
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,  # We want test data
    transform=transforms.ToTensor(),
    download=True
)

print(f"Downloaded {len(test_dataset)} test images")
print(f"Each image is 28×28 pixels")

In [ ]:
# Let's look at some sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    # Get a random image
    idx = np.random.randint(0, len(test_dataset))
    image, label = test_dataset[idx]
    
    # Display it
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')

plt.suptitle('Sample MNIST Digits', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Let's examine one image in detail
image, label = test_dataset[0]

print(f"Image shape: {image.shape}")
print(f"Data type: {image.dtype}")
print(f"Min pixel value: {image.min():.3f}")
print(f"Max pixel value: {image.max():.3f}")
print(f"Label: {label}")

# Show the raw pixel values
plt.figure(figsize=(8, 8))
plt.imshow(image.squeeze(), cmap='gray')
plt.colorbar(label='Pixel Value')
plt.title(f'MNIST Digit: {label}')
plt.show()

## Part 3: Define Our Neural Network

This is a simple neural network with:
- Input: 784 pixels (28×28 flattened)
- Hidden Layer 1: 128 neurons
- Hidden Layer 2: 64 neurons  
- Output: 10 neurons (one for each digit 0-9)

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        # Define layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        # This defines how data flows through the network
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

# Create an instance of our network
model = SimpleNN()
print("Neural Network Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Part 4: Load Pre-trained Weights

**For the actual lab**: I'll provide you with a pre-trained model file called `mnist_model.pth`

For this demo, we'll create a simple trained model:

In [ ]:
# FOR DEMO ONLY: Create a quick trained model
# In the real lab, you'll load a properly trained model

# Note: This is just for demonstration - don't worry about this code!
def quick_train_model(model, num_epochs=2):
    train_loader = torch.utils.data.DataLoader(
        torchvision.datasets.MNIST('./data', train=True, transform=transforms.ToTensor()),
        batch_size=64, shuffle=True
    )
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    model.train()
    for epoch in range(num_epochs):
        for i, (images, labels) in enumerate(train_loader):
            if i > 100:  # Just train on a subset for demo
                break
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model

print("Training a simple model for demo...")
model = quick_train_model(model)
print(" Model ready")

# Save the model (in real lab, you'll load this instead)
torch.save(model.state_dict(), 'mnist_model.pth')

In [ ]:
# This is what you'll actually do in the lab:
# Load the pre-trained model
model = SimpleNN()
model.load_state_dict(torch.load('mnist_model.pth'))
model.eval()  # Set to evaluation mode

print("Pre-trained model loaded successfully")

## Part 5: Test the Model

Now let's see how well our model recognizes digits!

In [ ]:
def predict_digit(model, image):
    """
    Predict what digit is in the image
    Returns: predicted digit and confidence scores
    """
    with torch.no_grad():  # We're not training, so no gradients needed
        # Add batch dimension
        image_batch = image.unsqueeze(0)
        
        # Get model output
        output = model(image_batch)
        
        # Convert to probabilities
        probabilities = torch.softmax(output, dim=1)
        
        # Get prediction
        predicted_digit = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, predicted_digit].item()
        
    return predicted_digit, probabilities.squeeze().numpy()

# Test on a random image
idx = np.random.randint(0, len(test_dataset))
image, true_label = test_dataset[idx]
predicted, probs = predict_digit(model, image)

# Visualize the result
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Show image
ax1.imshow(image.squeeze(), cmap='gray')
ax1.set_title(f'True Label: {true_label}')
ax1.axis('off')

# Show predictions
colors = ['blue'] * 10
colors[predicted] = 'green' if predicted == true_label else 'red'
bars = ax2.bar(range(10), probs, color=colors)
ax2.set_xlabel('Digit')
ax2.set_ylabel('Probability')
ax2.set_title(f'Predicted: {predicted} (Confidence: {probs[predicted]:.1%})')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Test on multiple images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i in range(5):
    # Get random image
    idx = np.random.randint(0, len(test_dataset))
    image, true_label = test_dataset[idx]
    predicted, probs = predict_digit(model, image)
    
    # Image
    axes[0, i].imshow(image.squeeze(), cmap='gray')
    axes[0, i].set_title(f'True: {true_label}')
    axes[0, i].axis('off')
    
    # Prediction bar chart
    colors = ['lightblue'] * 10
    colors[predicted] = 'green' if predicted == true_label else 'red'
    axes[1, i].bar(range(10), probs, color=colors)
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_title(f'Pred: {predicted}')
    
    # Top 3 predictions
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_text = "\n".join([f"{idx}: {probs[idx]:.1%}" for idx in top3_idx])
    axes[2, i].text(0.5, 0.5, top3_text, ha='center', va='center', fontsize=12)
    axes[2, i].set_title('Top 3')
    axes[2, i].axis('off')

plt.suptitle('Model Predictions on Random Test Images', fontsize=16)
plt.tight_layout()
plt.show()

## Part 6: Understanding Model Outputs

Let's look at what the neural network actually outputs:

In [ ]:
# Get one test image
image, label = test_dataset[0]

# Get raw model output (before softmax)
with torch.no_grad():
    raw_output = model(image.unsqueeze(0))
    probabilities = torch.softmax(raw_output, dim=1)

print("Raw model output (logits):")
print(raw_output.squeeze().numpy())
print("\nAfter softmax (probabilities):")
print(probabilities.squeeze().numpy())
print(f"\nSum of probabilities: {probabilities.sum().item():.6f}")
print(f"Predicted digit: {torch.argmax(probabilities).item()}")
print(f"True digit: {label}")

## Part 7: When Does the Model Fail?

Let's find examples where the model makes mistakes:

In [ ]:
# Find misclassified examples
misclassified = []
low_confidence = []

for i in range(min(1000, len(test_dataset))):
    image, true_label = test_dataset[i]
    predicted, probs = predict_digit(model, image)
    
    if predicted != true_label:
        misclassified.append((image, true_label, predicted, probs))
    
    if probs[predicted] < 0.9:  # Low confidence
        low_confidence.append((image, true_label, predicted, probs))

print(f"Found {len(misclassified)} misclassified images out of 1000")
print(f"Found {len(low_confidence)} low confidence predictions")

# Show some mistakes
if misclassified:
    fig, axes = plt.subplots(2, min(5, len(misclassified)), figsize=(15, 6))
    if len(misclassified) == 1:
        axes = axes.reshape(-1, 1)
    
    for i in range(min(5, len(misclassified))):
        image, true_label, predicted, probs = misclassified[i]
        
        # Show image
        axes[0, i].imshow(image.squeeze(), cmap='gray')
        axes[0, i].set_title(f'True: {true_label}, Pred: {predicted}')
        axes[0, i].axis('off')
        
        # Show probabilities
        colors = ['lightblue'] * 10
        colors[predicted] = 'red'
        colors[true_label] = 'green'
        axes[1, i].bar(range(10), probs, color=colors)
        axes[1, i].set_ylim(0, 1)
    
    plt.suptitle('Examples of Misclassified Digits', fontsize=16)
    plt.tight_layout()
    plt.show()

## Part 8: Model Performance Analysis

In [ ]:
# Calculate overall accuracy
correct = 0
total = 0
confusion_matrix = np.zeros((10, 10), dtype=int)

# Test on first 1000 images
for i in range(min(1000, len(test_dataset))):
    image, true_label = test_dataset[i]
    predicted, _ = predict_digit(model, image)
    
    if predicted == true_label:
        correct += 1
    total += 1
    
    confusion_matrix[true_label, predicted] += 1

accuracy = correct / total
print(f"Model Accuracy: {accuracy:.1%}")
print(f"Correct: {correct}/{total}")

# Show confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(confusion_matrix, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')

# Add numbers
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(confusion_matrix[i, j]), 
                ha='center', va='center',
                color='white' if confusion_matrix[i, j] > 50 else 'black')

plt.show()

# Most confused pairs
print("\nMost Confused Pairs:")
for i in range(10):
    for j in range(10):
        if i != j and confusion_matrix[i, j] > 5:
            print(f"{i} misclassified as {j}: {confusion_matrix[i, j]} times")

## Part 9: Key Takeaways

### What We Learned:

1. **MNIST Data**: 28×28 grayscale images of handwritten digits
2. **Neural Network Structure**: Input → Hidden layers → Output
3. **Model Output**: 10 values (one per digit), converted to probabilities
4. **Predictions**: The digit with highest probability wins
5. **Common Mistakes**: 4↔9, 3↔5, 7↔1 are often confused

### Questions to Think About:

- Why might the model confuse certain digit pairs?
- How could we improve the model's accuracy?
- What happens if we draw a digit differently than the training data?
- How is this different from traditional computer vision (Week 1-3)?